In [1]:
import cv2
import einops
import matplotlib.pyplot as plt
import mediapy
import numpy as np

import jax.numpy as jnp

from openpi.policies.libero_reason_dataset import LiberoSkillReasonDataset
from openpi.training import config as _config

In [2]:
data_config = _config.get_config('pi05_libero_skill_reason_lora_v2')
dataset = LiberoSkillReasonDataset(data_config.data.base_config, data_config.model.action_horizon)

The dataset you requested (None) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=None
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



Resolving data files:   0%|          | 0/4338 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/163 [00:00<?, ?it/s]

Using new skill reasoning dataset


In [3]:
import os
from pathlib import Path
import sys
SCRIPT_DIR = Path("../py_script")
sys.path.append(str(SCRIPT_DIR))
from vlm_interfaces import *

In [10]:
from vla_verify.scene_graph import TaskSceneGraph
PDDL_PATH = SCRIPT_DIR / "pddl" / "pick_place_domain.pddl"
pddl_domain_text = open(PDDL_PATH).read()
%env OPENROUTER_API_KEY=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
llm_interface, vlm_interface = get_openrouter_interfaces()

scene_graph = TaskSceneGraph(pddl_domain_text, vlm_interface)

env: OPENROUTER_API_KEY=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
Using OpenRouter
  LLM: google/gemini-2.5-flash
  VLM: google/gemini-2.5-pro


In [8]:
def image_tensor_to_cv2(image, resolution=(512,512)):
    return cv2.resize(np.array(einops.rearrange(image, "c h w -> h w c") * 255, dtype=np.uint8), resolution, interpolation=cv2.INTER_LANCZOS4)

def get_episode(episode_idx):
    reasonings = dataset.reasoning[episode_idx]
    start_idx = dataset.episode_starts[episode_idx]
    end_idx = dataset.episode_ends[episode_idx]
    data = dataset.hf_dataset[int(start_idx)]
    video_frames = []
    for i in range(start_idx, end_idx):
        img_data = dataset.hf_dataset[i]['image']
        video_frames.append(image_tensor_to_cv2(img_data))
    return reasonings, video_frames

reasonings, video_frames = get_episode(439)
mediapy.write_video(f'sample.mp4', video_frames, fps=20)

In [7]:
def _nl_task_to_pddl(llm_response, avail_actions

def process_episode(episode):
    reasonings, video_frames = episode
    scene_graph.read_image(video_frames, hint=f"The robot is trying to {reasonings['segments'][0]['instruction']}", ground=True)
    # results = scene_graph.ground_video(additional_points_labels=[
    #     ("robot", [[255, 90]])
    # ])
    for segment in reasonings['segments'][1:]:
        pass


In [11]:
process_episode((reasonings, video_frames))

  [VLM] Querying VLM for scene graph construction...


INFO 2026-03-07 01:25:53,660 1679099 sam3_video_predictor.py: 302: using the following GPU IDs: [0]
INFO 2026-03-07 01:25:53,661 1679099 sam3_video_predictor.py: 318: 


	*** START loading model on all ranks ***


INFO 2026-03-07 01:25:53,662 1679099 sam3_video_predictor.py: 320: loading model on rank=0 with world_size=1 -- this could take a while ...


  [VLM] Time elapsed: 12.292746739985887


INFO 2026-03-07 01:26:02,828 1679099 sam3_video_base.py: 125: setting max_num_objects=10000 and num_obj_for_compile=16
INFO 2026-03-07 01:26:05,692 1679099 sam3_video_predictor.py: 322: loading model on rank=0 with world_size=1 -- DONE locally
INFO 2026-03-07 01:26:05,693 1679099 sam3_video_predictor.py: 333: 


	*** DONE loading model on all ranks ***




Grounding objects:
0 a brown wooden table in the foreground
1 a wooden tray on the left side of the table, in the foreground
2 a silver bowl on the left side of the table, in the foreground
3 a silver bowl on the right side of the table, in the foreground
4 a bottle of salad dressing in the middle of the table, in the foreground
5 a small rectangular book on the right side of the table, in the foreground
7 tracks.
6 objects.
Matching: {4: 'bottle_of_salad_dressing_in_the_middle_of_the_table,_in_the_foreground_0', 5: 'small_rectangular_book_on_the_right_side_of_the_table,_in_the_foreground_0', 3: 'silver_bowl_on_the_right_side_of_the_table,_in_the_foreground_0', 2: 'silver_bowl_on_the_right_side_of_the_table,_in_the_foreground_1', 1: 'brown_wooden_table_in_the_foreground_2', 0: 'brown_wooden_table_in_the_foreground_4'}
Populating SAM3 cache...

...

Done.
Propagating detections...

  0%|          | 0/233 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Done.


/home/hppeng/workspace/mujoco_test/thirdparty/sam3/sam3/visualization_utils.py:185: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=figsize)


In [13]:
print(scene_graph.simulator)

AttributeError: 'ForallCondition' object has no attribute 'left_side'

In [34]:
import inspect
print(inspect.getsource(scene_graph._read_image))

    def _read_image(self, llm_response, images, ground=True):
        """
        Read an RGB image with a VLM, output a PDDL domain file and parse that into a scene graph.
        """
        if type(images) == list:
            image_rgb = images[0]
        else:
            image_rgb = images
            images = [image_rgb]
        print(type(image_rgb))
        input()
        t0 = time.monotonic()
        print(f"  [VLM] Querying VLM for scene graph construction...")
        yield [ transformers_api.make_message(images=[image_rgb]) ]
        t1 = time.monotonic()
        print(f"  [VLM] Time elapsed: {t1 - t0}")

        # Strip triple backticks
        raw_response = llm_response['content']
        response = extract_in_backticks(raw_response, 'pddl')
        if response is None:
            print("Warning: No triple backticks given")
            response = raw_response

        yield self.construct_from_pddl(images, response, ground=ground)

